# Audio CNN Model Comparison VF

Training-only notebook for six-class audio event classification. It compares one scratch CNN, the final VF ResNet34 spectrogram model, and three lightweight candidate pretrained branches. The goal is architecture comparison before final tuning.

## Coverage

- Data collection and preprocessing
- Train-only augmentation
- Log-mel normalization and 224 x 224 resizing
- 70 / 15 / 15 stratified split
- Scratch CNN baseline
- Final VF ResNet34 spectrogram model
- PANNs Cnn14 audio embedding model
- MobileNetV3-Small spectrogram model
- EfficientNet-B0 spectrogram model
- Transfer learning with frozen and fine-tune phases
- Regularization, AdamW, scheduler, cross-entropy loss
- Accuracy, macro F1, weighted F1, AUC, ECE, latency, parameter count, model size
- Learning curves, confusion matrices, calibration, top confusions, per-class reports
- Grad-CAM, LIME-style spectrogram attribution, and waveform occlusion for PANNs
- Final production-candidate selection

## Dependencies

In [ ]:
import importlib.util, subprocess, sys
needed = [("librosa", "librosa"), ("soundfile", "soundfile"), ("seaborn", "seaborn"), ("tqdm", "tqdm"), ("scikit-learn", "sklearn")]
missing = [pkg for pkg, mod in needed if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
optional = [("torchlibrosa", "torchlibrosa"), ("panns-inference", "panns_inference")]
for pkg, mod in optional:
    if importlib.util.find_spec(mod) is None:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", pkg])
        except Exception as e:
            print(f"{pkg} unavailable: {e}")
print("Dependencies ready")

## Configuration

In [ ]:
import os
from pathlib import Path
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("TQDM_DISABLE", "1")
CFG = {
    "output_dir": "/kaggle/working/audio_model_comparison",
    "model_dir": "/kaggle/working/audio_model_comparison/models",
    "xai_dir": "/kaggle/working/audio_model_comparison/xai",
    "tmp_dir": "/tmp/audio_model_comparison_vf",
    "ds_esc50": "/kaggle/input/datasets/niranjankn/esc50/ESC-50-master",
    "ds_urban": "/kaggle/input/datasets/chrisfilo/urbansound8k",
    "ds_ravdess": "/kaggle/input/datasets/dejolilandry/ravdess/Radvess",
    "ds_cremad": "/kaggle/input/datasets/ejlok1/cremad/AudioWAV",
    "ds_gunshot": "/kaggle/input/datasets/emrahaydemr/gunshot-audio-dataset",
    "ds_siren": "/kaggle/input/datasets/vishnu0399/emergency-vehicle-siren-sounds/sounds",
    "ds_babycry": "/kaggle/input/datasets/warcoder/infant-cry-audio-corpus/donateacry_corpus",
    "ds_music": "/kaggle/input/datasets/lnicalo/gtzan-musicspeech-collection/music_wav",
    "sample_rate": 32000,
    "segment_duration": 3.0,
    "n_mels": 64,
    "image_size": 224,
    "max_per_class": 900,
    "glass_break_augment_target": 350,
    "baby_cry_augment_target": 500,
    "batch_size": 32,
    "num_workers": 0,
    "scratch_epochs": 5,
    "pretrained_frozen_epochs": 2,
    "candidate_full_epochs": 3,
    "final_vf_full_epochs": 6,
    "scratch_lr": 1e-3,
    "head_lr": 1e-3,
    "full_lr": 1e-4,
    "weight_decay": 1e-3,
    "label_smoothing": 0.05,
    "mixup_alpha": 0.20,
    "early_stop_patience": 2,
    "xai_examples": 1,
    "lime_masks": 60,
    "show_progress": False,
    "quiet_external_output": True,
    "seed": 42
}
for key in ["output_dir", "model_dir", "xai_dir", "tmp_dir"]:
    Path(CFG[key]).mkdir(parents=True, exist_ok=True)
print("Config loaded")

## Imports and Labels

In [ ]:
import gc, csv, json, math, time, random, hashlib, warnings, contextlib, sys, os
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import librosa, soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import Ridge
from IPython.display import display, Markdown, Image
warnings.filterwarnings("ignore")
@contextlib.contextmanager
def mute_output(enabled=True):
    if not enabled:
        yield
        return
    with open(os.devnull, "w") as devnull:
        old_stdout = os.dup(1)
        old_stderr = os.dup(2)
        try:
            os.dup2(devnull.fileno(), 1)
            os.dup2(devnull.fileno(), 2)
            with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
                yield
        finally:
            os.dup2(old_stdout, 1)
            os.dup2(old_stderr, 2)
            os.close(old_stdout)
            os.close(old_stderr)
SEED = CFG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_CLASSES = ["gunshot", "glass_break", "alarm_signal", "human_voice", "baby_cry", "background"]
CLASS2IDX = {c: i for i, c in enumerate(TARGET_CLASSES)}
IDX2CLASS = {i: c for c, i in CLASS2IDX.items()}
NUM_CLASSES = len(TARGET_CLASSES)
print(f"Device: {DEVICE}")
print(f"Classes: {TARGET_CLASSES}")

## Data Collection and Preprocessing

In [ ]:
ESC50_MAP = {
    "glass_breaking": "glass_break", "siren": "alarm_signal", "clock_alarm": "alarm_signal", "car_horn": "alarm_signal",
    "crying_baby": "baby_cry", "laughing": "human_voice", "coughing": "human_voice", "sneezing": "human_voice",
    "clapping": "human_voice", "breathing": "human_voice", "snoring": "human_voice",
    "dog": "background", "footsteps": "background", "drinking_sipping": "background", "brushing_teeth": "background",
    "keyboard_typing": "background", "mouse_click": "background", "door_wood_knock": "background", "door_wood_creaks": "background",
    "can_opening": "background", "pouring_water": "background", "toilet_flush": "background", "washing_machine": "background",
    "vacuum_cleaner": "background", "clock_tick": "background", "engine": "background", "helicopter": "background",
    "chainsaw": "background", "hand_saw": "background", "airplane": "background", "train": "background",
    "fireworks": "background", "church_bells": "background", "crackling_fire": "background", "thunderstorm": "background",
    "rain": "background", "sea_waves": "background", "wind": "background", "water_drops": "background",
    "insects": "background", "crickets": "background", "cat": "background", "rooster": "background",
    "hen": "background", "frog": "background", "crow": "background", "cow": "background", "pig": "background",
    "sheep": "background", "chirping_birds": "background"
}
URBAN_MAP = {
    "gun_shot": "gunshot", "siren": "alarm_signal", "car_horn": "alarm_signal", "children_playing": "human_voice",
    "dog_bark": "background", "street_music": "background", "air_conditioner": "background", "engine_idling": "background",
    "drilling": "background", "jackhammer": "background"
}
RAVDESS_MAP = {f"{i:02d}": "human_voice" for i in range(1, 9)}
CREMAD_MAP = {e: "human_voice" for e in ["NEU", "HAP", "SAD", "ANG", "DIS", "FEA"]}

def scan_esc50():
    root = Path(CFG["ds_esc50"])
    csv_path = root / "meta" / "esc50.csv"
    audio_dir = root / "audio"
    if not csv_path.exists():
        return []
    out = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            label = ESC50_MAP.get(row["category"])
            path = audio_dir / row["filename"]
            if label and path.exists():
                out.append((str(path), label, "esc50"))
    return out

def scan_urbansound8k():
    root = Path(CFG["ds_urban"])
    csv_path = root / "UrbanSound8K.csv"
    if not csv_path.exists():
        return []
    out = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            label = URBAN_MAP.get(row["class"])
            path = root / f"fold{row['fold']}" / row["slice_file_name"]
            if label and path.exists():
                out.append((str(path), label, "urban"))
    return out

def scan_ravdess():
    root = Path(CFG["ds_ravdess"])
    if not root.exists():
        return []
    out = []
    for actor in sorted(root.iterdir()):
        if actor.is_dir():
            for path in actor.glob("*.wav"):
                parts = path.stem.split("-")
                if len(parts) >= 3:
                    label = RAVDESS_MAP.get(parts[2])
                    if label:
                        out.append((str(path), label, "ravdess"))
    return out

def scan_cremad():
    root = Path(CFG["ds_cremad"])
    if not root.exists():
        return []
    out = []
    for path in root.glob("*.wav"):
        parts = path.stem.split("_")
        if len(parts) >= 3:
            label = CREMAD_MAP.get(parts[2])
            if label:
                out.append((str(path), label, "cremad"))
    return out

def scan_gunshot():
    root = Path(CFG["ds_gunshot"])
    return [(str(p), "gunshot", "gunshot") for p in root.rglob("*.wav")] if root.exists() else []

def scan_siren():
    root = Path(CFG["ds_siren"])
    if not root.exists():
        return []
    out = []
    for name in ["ambulance", "firetruck"]:
        folder = root / name
        if folder.exists():
            out.extend((str(p), "alarm_signal", "siren_ds") for p in folder.glob("*.wav"))
    traffic = root / "traffic"
    if traffic.exists():
        out.extend((str(p), "background", "siren_traffic") for p in traffic.glob("*.wav"))
    return out

def scan_babycry():
    root = Path(CFG["ds_babycry"])
    if not root.exists():
        return []
    out = []
    for folder in sorted(root.iterdir()):
        if folder.is_dir():
            out.extend((str(p), "baby_cry", "babycry") for p in folder.glob("*.wav"))
    return out

def scan_music():
    root = Path(CFG["ds_music"])
    return [(str(p), "background", "music") for p in root.glob("*.wav")] if root.exists() else []

def fingerprint_audio(path):
    try:
        y, _ = librosa.load(path, sr=8000, mono=True, duration=3.0)
        return hashlib.sha256((np.clip(y, -1, 1) * 127).astype(np.int8).tobytes()).hexdigest()[:16]
    except Exception:
        return None

def deduplicate(samples):
    seen, out = set(), []
    for item in samples:
        fp = fingerprint_audio(item[0])
        if fp is None or fp not in seen:
            if fp is not None:
                seen.add(fp)
            out.append(item)
    print(f"Dedup: {len(samples)} -> {len(out)}")
    return out

def cap_per_class(samples, cap):
    buckets = defaultdict(list)
    rng = random.Random(SEED)
    for item in samples:
        buckets[item[1]].append(item)
    out = []
    for cls in TARGET_CLASSES:
        rows = buckets.get(cls, [])
        rng.shuffle(rows)
        out.extend(rows[:cap] if cap else rows)
    rng.shuffle(out)
    return out

def prepare_splits(samples):
    labels = [s[1] for s in samples]
    strat = [f"{s[1]}_{s[2]}" for s in samples]
    counts = Counter(strat)
    safe = [st if counts[st] >= 3 else labels[i] for i, st in enumerate(strat)]
    train_val, test = train_test_split(samples, test_size=0.15, stratify=safe, random_state=SEED)
    train, val = train_test_split(train_val, test_size=0.15 / 0.85, stratify=[s[1] for s in train_val], random_state=SEED)
    return train, val, test

def augment_minority(samples, cls, target):
    src = [s for s in samples if s[1] == cls]
    if not src or len(src) >= target:
        return samples
    out_dir = Path(CFG["tmp_dir"]) / "aug"
    out_dir.mkdir(parents=True, exist_ok=True)
    aug = []
    i = 0
    while len(src) + len(aug) < target and i < target * 8:
        path, label, source = src[i % len(src)]
        try:
            y, _ = librosa.load(path, sr=CFG["sample_rate"], mono=True)
            mode = i // len(src)
            if mode % 4 == 0:
                y = librosa.effects.pitch_shift(y, sr=CFG["sample_rate"], n_steps=np.random.uniform(-2.0, 2.0))
            elif mode % 4 == 1:
                y = librosa.effects.time_stretch(y, rate=np.random.uniform(0.9, 1.1))
            elif mode % 4 == 2:
                y = y + np.random.normal(0, 0.005, len(y)).astype(np.float32)
            else:
                y = (y * np.random.uniform(0.75, 1.25)).astype(np.float32)
            out_path = out_dir / f"{cls}_{i}.wav"
            sf.write(out_path, y, CFG["sample_rate"])
            aug.append((str(out_path), label, source + "_aug"))
        except Exception:
            pass
        i += 1
    print(f"Augmented {cls}: {len(src)} -> {len(src) + len(aug)}")
    return samples + aug

def class_weights(samples):
    counts = Counter(s[1] for s in samples)
    total = sum(counts.values())
    weights = torch.zeros(NUM_CLASSES)
    for cls, idx in CLASS2IDX.items():
        weights[idx] = math.sqrt(total / (NUM_CLASSES * max(1, counts.get(cls, 1))))
    weights = weights / weights.sum() * NUM_CLASSES
    return weights

## Dataset Audit and 70 / 15 / 15 Split

In [ ]:
scanners = [("ESC-50", scan_esc50), ("UrbanSound8K", scan_urbansound8k), ("RAVDESS", scan_ravdess), ("CREMA-D", scan_cremad), ("Gunshot", scan_gunshot), ("Siren", scan_siren), ("BabyCry", scan_babycry), ("Music", scan_music)]
all_samples = []
source_counts = {}
for name, fn in scanners:
    rows = fn()
    source_counts[name] = len(rows)
    all_samples.extend(rows)
    print(f"{name:14s}: {len(rows)}")
if not all_samples:
    raise RuntimeError("No audio datasets found")
raw_counts = dict(sorted(Counter(s[1] for s in all_samples).items()))
all_samples = deduplicate(all_samples)
dedup_counts = dict(sorted(Counter(s[1] for s in all_samples).items()))
all_samples = cap_per_class(all_samples, CFG["max_per_class"])
capped_counts = dict(sorted(Counter(s[1] for s in all_samples).items()))
train_s, val_s, test_s = prepare_splits(all_samples)
train_s = augment_minority(train_s, "glass_break", CFG["glass_break_augment_target"])
train_s = augment_minority(train_s, "baby_cry", CFG["baby_cry_augment_target"])
split_counts = {"train": dict(sorted(Counter(s[1] for s in train_s).items())), "val": dict(sorted(Counter(s[1] for s in val_s).items())), "test": dict(sorted(Counter(s[1] for s in test_s).items()))}
DATA_AUDIT = {"classes": TARGET_CLASSES, "source_counts": source_counts, "raw_counts": raw_counts, "dedup_counts": dedup_counts, "capped_counts": capped_counts, "split_counts": split_counts, "split_rule": "70/15/15 stratified, deduplicate before split, augment train only"}
Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)
with open(Path(CFG["output_dir"]) / "dataset_audit.json", "w", encoding="utf-8") as f:
    json.dump(DATA_AUDIT, f, indent=2)
print(json.dumps(split_counts, indent=2))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(x=list(capped_counts.values()), y=list(capped_counts.keys()), ax=axes[0])
axes[0].set_title("capped class distribution")
rows = []
for split, counts in split_counts.items():
    for cls, count in counts.items():
        rows.append({"split": split, "class": cls, "count": count})
sns.barplot(data=pd.DataFrame(rows), x="class", y="count", hue="split", ax=axes[1])
axes[1].tick_params(axis="x", rotation=35)
axes[1].set_title("train validation test split")
fig.tight_layout()
fig.savefig(Path(CFG["output_dir"]) / "dataset_split.png", dpi=160)
plt.show()

## Dataset and Loaders

In [ ]:
def load_fixed_audio(path):
    target = int(CFG["sample_rate"] * CFG["segment_duration"])
    y, _ = librosa.load(path, sr=CFG["sample_rate"], mono=True)
    if len(y) > target:
        y = y[:target]
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    return y.astype(np.float32)

def audio_to_spec(y):
    mel = librosa.feature.melspectrogram(y=y, sr=CFG["sample_rate"], n_mels=CFG["n_mels"], fmax=CFG["sample_rate"] // 2)
    db = librosa.power_to_db(mel, ref=np.max, top_db=80)
    spec = np.clip((db + 80) / 80, 0, 1).astype(np.float32)
    spec = torch.tensor(spec).unsqueeze(0)
    spec = F.interpolate(spec.unsqueeze(0), size=(CFG["image_size"], CFG["image_size"]), mode="bilinear", align_corners=False).squeeze(0)
    return spec

class AudioComparisonDataset(Dataset):
    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label, source = self.samples[idx]
        try:
            y = load_fixed_audio(path)
        except Exception:
            y = np.zeros(int(CFG["sample_rate"] * CFG["segment_duration"]), dtype=np.float32)
        if self.augment:
            if np.random.random() < 0.35:
                y = (y * np.random.uniform(0.75, 1.25)).astype(np.float32)
            if np.random.random() < 0.25:
                y = (y + np.random.normal(0, 0.003, len(y))).astype(np.float32)
        spec = audio_to_spec(y)
        wave = torch.tensor(y, dtype=torch.float32)
        return spec, wave, CLASS2IDX[label], path, source

def collate_batch(batch):
    specs = torch.stack([x[0] for x in batch])
    waves = torch.stack([x[1] for x in batch])
    labels = torch.tensor([x[2] for x in batch], dtype=torch.long)
    paths = [x[3] for x in batch]
    sources = [x[4] for x in batch]
    return specs, waves, labels, paths, sources

train_dl = DataLoader(AudioComparisonDataset(train_s, augment=True), batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"], pin_memory=True, drop_last=True, collate_fn=collate_batch)
val_dl = DataLoader(AudioComparisonDataset(val_s), batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=True, collate_fn=collate_batch)
test_dl = DataLoader(AudioComparisonDataset(test_s), batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=True, collate_fn=collate_batch)
CLASS_WEIGHTS = class_weights(train_s).to(DEVICE)
print(f"train={len(train_s)} val={len(val_s)} test={len(test_s)}")

## Modelling Objective

The task is supervised six-class audio event classification. The notebook compares a scratch CNN baseline, the final VF ResNet34 candidate, and three lightweight pretrained branches under the same data split and evaluation protocol. The final VF branch receives the slightly stronger production-candidate fine-tuning budget; the comparison remains metric-driven and does not hardcode the winner.

In [ ]:
def find_last_conv(module):
    last = None
    for layer in module.modules():
        if isinstance(layer, nn.Conv2d):
            last = layer
    return last

class ScratchAudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 192, 3, padding=1), nn.BatchNorm2d(192), nn.ReLU(), nn.AdaptiveAvgPool2d(1)
        )
        self.cam_layer = self.features[12]
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.45), nn.Linear(192, 128), nn.ReLU(), nn.Dropout(0.25), nn.Linear(128, NUM_CLASSES))
    def forward(self, spec, wave=None):
        return self.head(self.features(spec))
    def freeze_backbone(self):
        return
    def unfreeze_backbone(self):
        return

class ResNet34Spectrogram(nn.Module):
    def __init__(self):
        super().__init__()
        try:
            with mute_output(CFG["quiet_external_output"]):
                self.backbone = models.resnet34(weights=models.ResNet34_Weights.DEFAULT, progress=False)
        except Exception:
            self.backbone = models.resnet34(weights=None, progress=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.head = nn.Sequential(nn.Dropout(0.35), nn.Linear(in_features, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.25), nn.Linear(256, NUM_CLASSES))
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.cam_layer = find_last_conv(self.backbone)
    def forward(self, spec, wave=None):
        x = spec.repeat(1, 3, 1, 1)
        x = (x - self.mean) / self.std
        return self.head(self.backbone(x))
    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

class TorchvisionSpectrogram(nn.Module):
    def __init__(self, arch):
        super().__init__()
        self.arch = arch
        try:
            with mute_output(CFG["quiet_external_output"]):
                if arch == "mobilenet_v3_small":
                    self.backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT, progress=False)
                elif arch == "efficientnet_b0":
                    self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT, progress=False)
                else:
                    raise ValueError(arch)
        except Exception:
            if arch == "mobilenet_v3_small":
                self.backbone = models.mobilenet_v3_small(weights=None, progress=False)
            elif arch == "efficientnet_b0":
                self.backbone = models.efficientnet_b0(weights=None, progress=False)
            else:
                raise
        if arch == "mobilenet_v3_small":
            in_features = self.backbone.classifier[-1].in_features
            self.backbone.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)
        elif arch == "efficientnet_b0":
            in_features = self.backbone.classifier[-1].in_features
            self.backbone.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)
        self.cam_layer = find_last_conv(self.backbone)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, spec, wave=None):
        x = spec.repeat(1, 3, 1, 1)
        x = (x - self.mean) / self.std
        return self.backbone(x)
    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False
        if hasattr(self.backbone, "classifier"):
            for p in self.backbone.classifier.parameters():
                p.requires_grad = True
    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

class PANNsCnn14Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        from panns_inference import AudioTagging
        with mute_output(CFG["quiet_external_output"]):
            audio_tagger = AudioTagging(checkpoint_path=None, device=str(DEVICE))
        self.backbone = audio_tagger.model
        with torch.no_grad():
            out = self.backbone(torch.zeros(1, int(CFG["sample_rate"] * CFG["segment_duration"])).to(DEVICE))
        if isinstance(out, dict):
            emb = out.get("embedding", list(out.values())[0])
        elif isinstance(out, tuple):
            emb = out[0]
        else:
            emb = out
        if emb.dim() > 2:
            emb = emb.mean(dim=tuple(range(1, emb.dim() - 1)))
        dim = emb.shape[-1]
        self.head = nn.Sequential(nn.Dropout(0.35), nn.Linear(dim, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.25), nn.Linear(256, NUM_CLASSES))
    def features(self, wave):
        out = self.backbone(wave)
        if isinstance(out, dict):
            emb = out.get("embedding", list(out.values())[0])
        elif isinstance(out, tuple):
            emb = out[0]
        else:
            emb = out
        if emb.dim() > 2:
            emb = emb.mean(dim=tuple(range(1, emb.dim() - 1)))
        return emb
    def forward(self, spec, wave=None):
        return self.head(self.features(wave))
    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False
        for p in self.head.parameters():
            p.requires_grad = True

def build_models():
    registry = {
        "scratch_cnn": ScratchAudioCNN(),
        "final_vf_resnet34": ResNet34Spectrogram(),
        "candidate_mobilenet_v3_small": TorchvisionSpectrogram("mobilenet_v3_small"),
        "candidate_efficientnet_b0": TorchvisionSpectrogram("efficientnet_b0")
    }
    try:
        registry["candidate_panns_cnn14"] = PANNsCnn14Classifier()
    except Exception as e:
        print(f"candidate_panns_cnn14 unavailable: {e}")
    return registry

MODEL_REGISTRY = build_models()
print(list(MODEL_REGISTRY.keys()))

## Training and Evaluation Utilities

In [ ]:
def count_params(model):
    return int(sum(p.numel() for p in model.parameters()))

def model_size_mb(path):
    return round(Path(path).stat().st_size / (1024 * 1024), 3) if Path(path).exists() else 0.0

def mixup(spec, wave, labels, alpha):
    if alpha <= 0:
        return spec, wave, labels, labels, 1.0
    lam = max(np.random.beta(alpha, alpha), 0.5)
    idx = torch.randperm(spec.size(0), device=spec.device)
    return lam * spec + (1 - lam) * spec[idx], lam * wave + (1 - lam) * wave[idx], labels, labels[idx], lam

def train_loss(criterion, logits, ya, yb, lam):
    return lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)

@torch.no_grad()
def collect_outputs(model, loader, temperature=1.0):
    model.eval()
    probs, preds, labels, paths, sources = [], [], [], [], []
    for specs, waves, y, p, s in loader:
        specs = specs.to(DEVICE)
        waves = waves.to(DEVICE)
        logits = model(specs, waves) / max(float(temperature), 1e-6)
        pr = F.softmax(logits, dim=1).cpu().numpy()
        probs.append(pr)
        preds.extend(pr.argmax(axis=1).tolist())
        labels.extend(y.numpy().tolist())
        paths.extend(p)
        sources.extend(s)
    probs = np.concatenate(probs, axis=0)
    return {"probs": probs, "preds": np.array(preds), "labels": np.array(labels), "paths": paths, "sources": sources}

@torch.no_grad()
def macro_f1_on_loader(model, loader):
    out = collect_outputs(model, loader)
    return f1_score(out["labels"], out["preds"], average="macro", zero_division=0)

def tune_temperature(model, loader):
    model.eval()
    logits_list, labels_list = [], []
    with torch.no_grad():
        for specs, waves, y, _, _ in loader:
            logits_list.append(model(specs.to(DEVICE), waves.to(DEVICE)))
            labels_list.append(y.to(DEVICE))
    logits = torch.cat(logits_list)
    labels = torch.cat(labels_list)
    best_t, best_loss = 1.0, float("inf")
    for t in np.linspace(0.6, 3.0, 25):
        loss = F.cross_entropy(logits / float(t), labels).item()
        if loss < best_loss:
            best_t, best_loss = float(t), float(loss)
    return best_t

def ece_score(labels, probs, bins=10):
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == labels).astype(float)
    ece = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf >= lo) & (conf < hi if hi < 1 else conf <= hi)
        if mask.any():
            ece += float(mask.mean()) * abs(float(correct[mask].mean()) - float(conf[mask].mean()))
    return round(ece, 5)

def metric_dict(name, model, output, temperature, history, path, latency_ms):
    labels, preds, probs = output["labels"], output["preds"], output["probs"]
    try:
        auc = roc_auc_score(labels, probs, labels=list(range(NUM_CLASSES)), multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    return {
        "model": name,
        "accuracy": round(float(accuracy_score(labels, preds)), 5),
        "macro_f1": round(float(f1_score(labels, preds, average="macro", zero_division=0)), 5),
        "weighted_f1": round(float(f1_score(labels, preds, average="weighted", zero_division=0)), 5),
        "precision_macro": round(float(precision_score(labels, preds, average="macro", zero_division=0)), 5),
        "recall_macro": round(float(recall_score(labels, preds, average="macro", zero_division=0)), 5),
        "auc_macro_ovr": None if np.isnan(auc) else round(float(auc), 5),
        "ece": ece_score(labels, probs),
        "temperature": round(float(temperature), 4),
        "parameters": count_params(model),
        "model_mb": model_size_mb(path),
        "latency_ms": round(float(latency_ms), 3),
        "best_val_f1": round(float(max([h["val_f1"] for h in history] or [0])), 5)
    }

def measure_latency(model, loader, repeats=15):
    model.eval()
    specs, waves, _, _, _ = next(iter(loader))
    specs = specs.to(DEVICE)
    waves = waves.to(DEVICE)
    with torch.no_grad():
        for _ in range(3):
            model(specs, waves)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(repeats):
            model(specs, waves)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    return 1000 * (time.time() - start) / repeats

## Train Scratch and Pretrained Branches

In [ ]:
def train_model(name, model):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS, label_smoothing=CFG["label_smoothing"])
    history = []
    best_state = None
    best_val = -1.0
    patience = 0
    if name == "scratch_cnn":
        phases = [("scratch", CFG["scratch_epochs"], CFG["scratch_lr"], True)]
    else:
        full_epochs = CFG["final_vf_full_epochs"] if name == "final_vf_resnet34" else CFG["candidate_full_epochs"]
        phases = [("frozen", CFG["pretrained_frozen_epochs"], CFG["head_lr"], False), ("full", full_epochs, CFG["full_lr"], True)]
    for phase, epochs, lr, train_backbone in phases:
        if train_backbone:
            model.unfreeze_backbone()
        else:
            model.freeze_backbone()
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=CFG["weight_decay"])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs * len(train_dl)))
        for ep in range(epochs):
            model.train()
            total_loss, total_correct, total = 0.0, 0, 0
            for specs, waves, labels, _, _ in tqdm(train_dl, desc=f"{name} {phase} {ep + 1}/{epochs}", leave=False, disable=not CFG["show_progress"]):
                specs = specs.to(DEVICE)
                waves = waves.to(DEVICE)
                labels = labels.to(DEVICE)
                mx, mw, ya, yb, lam = mixup(specs, waves, labels, CFG["mixup_alpha"])
                logits = model(mx, mw)
                loss = train_loss(criterion, logits, ya, yb, lam)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                total_loss += loss.item() * specs.size(0)
                total_correct += (logits.argmax(1) == labels).sum().item()
                total += specs.size(0)
            val_f1 = macro_f1_on_loader(model, val_dl)
            row = {"model": name, "phase": phase, "epoch": ep + 1, "loss": total_loss / max(1, total), "train_acc": total_correct / max(1, total), "val_f1": val_f1, "lr": scheduler.get_last_lr()[0]}
            history.append(row)
            print(f"{name} | {phase} {ep + 1}/{epochs} | loss={row['loss']:.4f} | acc={row['train_acc']:.4f} | val_f1={row['val_f1']:.4f} | lr={row['lr']:.2e}")
            if val_f1 > best_val:
                best_val = val_f1
                patience = 0
                best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            else:
                patience += 1
            if patience >= CFG["early_stop_patience"]:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    temperature = tune_temperature(model, val_dl)
    path = Path(CFG["model_dir"]) / f"{name}.pt"
    torch.save({"model_state": model.state_dict(), "classes": TARGET_CLASSES, "cfg": CFG, "history": history, "temperature": temperature}, path)
    test_output = collect_outputs(model, test_dl, temperature=temperature)
    latency = measure_latency(model, test_dl)
    metrics = metric_dict(name, model, test_output, temperature, history, path, latency)
    pd.DataFrame(history).to_csv(Path(CFG["output_dir"]) / f"{name}_history.csv", index=False)
    with open(Path(CFG["output_dir"]) / f"{name}_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    return {"model": model, "history": history, "output": test_output, "metrics": metrics, "path": str(path), "temperature": temperature}

RUNS = {}
for model_name, model_obj in MODEL_REGISTRY.items():
    print(f"\nTraining {model_name}")
    try:
        RUNS[model_name] = train_model(model_name, model_obj)
    except Exception as e:
        print(f"{model_name} failed: {e}")
        RUNS[model_name] = {"failed": str(e)}
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
valid_runs = {k: v for k, v in RUNS.items() if "metrics" in v}
comparison = pd.DataFrame([v["metrics"] for v in valid_runs.values()]).sort_values("macro_f1", ascending=False)
comparison.to_csv(Path(CFG["output_dir"]) / "model_comparison.csv", index=False)
with open(Path(CFG["output_dir"]) / "model_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison.to_dict("records"), f, indent=2)
display(Markdown("### Model comparison leaderboard"))
metric_cols = ["model", "macro_f1", "accuracy", "auc_macro_ovr", "ece", "latency_ms", "parameters", "model_mb", "temperature"]
visible_cols = [c for c in metric_cols if c in comparison.columns]
display(comparison[visible_cols].style.format({
    "macro_f1": "{:.4f}",
    "accuracy": "{:.4f}",
    "auc_macro_ovr": "{:.4f}",
    "ece": "{:.4f}",
    "latency_ms": "{:.2f}",
    "model_mb": "{:.2f}",
    "temperature": "{:.3f}"
}).background_gradient(subset=[c for c in ["macro_f1", "accuracy", "auc_macro_ovr"] if c in visible_cols], cmap="YlGn"))
BEST_MODEL_NAME = comparison.iloc[0]["model"] if len(comparison) else None
print(f"Best model for later tuning: {BEST_MODEL_NAME}")

## Evaluation and Side-by-Side Comparison

In [ ]:
def plot_history(name, history):
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    sns.lineplot(data=df, x=df.index + 1, y="loss", hue="phase", marker="o", ax=axes[0])
    sns.lineplot(data=df, x=df.index + 1, y="train_acc", hue="phase", marker="o", ax=axes[1])
    sns.lineplot(data=df, x=df.index + 1, y="val_f1", hue="phase", marker="o", ax=axes[2])
    axes[0].set_title(f"{name} loss")
    axes[1].set_title(f"{name} train accuracy")
    axes[2].set_title(f"{name} validation macro F1")
    for ax in axes:
        ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(Path(CFG["output_dir"]) / f"{name}_learning_curve.png", dpi=160)
    plt.show()

def plot_confusion(name, output):
    cm = confusion_matrix(output["labels"], output["preds"], labels=list(range(NUM_CLASSES)))
    cmn = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES, ax=axes[0])
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES, ax=axes[1])
    axes[0].set_title(f"{name} confusion matrix")
    axes[1].set_title(f"{name} normalized confusion matrix")
    for ax in axes:
        ax.set_xlabel("predicted")
        ax.set_ylabel("true")
    fig.tight_layout()
    fig.savefig(Path(CFG["output_dir"]) / f"{name}_confusion_matrix.png", dpi=160)
    plt.show()

def plot_calibration(name, output):
    probs = output["probs"]
    labels = output["labels"]
    preds = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    correct = (preds == labels).astype(float)
    bins = np.linspace(0, 1, 11)
    xs, ys = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (conf >= lo) & (conf < hi if hi < 1 else conf <= hi)
        if mask.any():
            xs.append(conf[mask].mean())
            ys.append(correct[mask].mean())
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(conf[preds == labels], bins=20, alpha=0.7, label="correct")
    axes[0].hist(conf[preds != labels], bins=20, alpha=0.7, label="wrong")
    axes[0].legend()
    axes[0].set_title(f"{name} confidence")
    axes[1].plot([0, 1], [0, 1], linestyle="--", color="black")
    axes[1].plot(xs, ys, marker="o")
    axes[1].set_title(f"{name} calibration")
    axes[1].set_xlabel("confidence")
    axes[1].set_ylabel("accuracy")
    fig.tight_layout()
    fig.savefig(Path(CFG["output_dir"]) / f"{name}_calibration.png", dpi=160)
    plt.show()

def report_frame(output):
    report = classification_report(output["labels"], output["preds"], target_names=TARGET_CLASSES, zero_division=0, output_dict=True)
    df = pd.DataFrame(report).T.reset_index().rename(columns={"index": "class"})
    for col in ["precision", "recall", "f1-score", "support"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def top_confusions_frame(output, limit=8):
    cm = confusion_matrix(output["labels"], output["preds"], labels=list(range(NUM_CLASSES)))
    rows = []
    for i, true_name in enumerate(TARGET_CLASSES):
        for j, pred_name in enumerate(TARGET_CLASSES):
            if i != j and cm[i, j] > 0:
                rows.append({"true": true_name, "predicted": pred_name, "count": int(cm[i, j])})
    return pd.DataFrame(rows).sort_values("count", ascending=False).head(limit) if rows else pd.DataFrame(columns=["true", "predicted", "count"])

display(Markdown("### Leaderboard"))
display(comparison[visible_cols].style.format({
    "macro_f1": "{:.4f}",
    "accuracy": "{:.4f}",
    "auc_macro_ovr": "{:.4f}",
    "ece": "{:.4f}",
    "latency_ms": "{:.2f}",
    "model_mb": "{:.2f}",
    "temperature": "{:.3f}"
}).background_gradient(subset=[c for c in ["macro_f1", "accuracy", "auc_macro_ovr"] if c in visible_cols], cmap="YlGn"))

for name, run in valid_runs.items():
    display(Markdown(f"### {name}"))
    report_df = report_frame(run["output"])
    report_path = Path(CFG["output_dir"]) / f"{name}_classification_report.csv"
    report_df.to_csv(report_path, index=False)
    display(report_df.style.format({"precision": "{:.4f}", "recall": "{:.4f}", "f1-score": "{:.4f}", "support": "{:.0f}"}))
    confusions_df = top_confusions_frame(run["output"])
    confusions_path = Path(CFG["output_dir"]) / f"{name}_top_confusions.csv"
    confusions_df.to_csv(confusions_path, index=False)
    display(Markdown("Top confusions"))
    display(confusions_df)
    plot_history(name, run["history"])
    plot_confusion(name, run["output"])
    plot_calibration(name, run["output"])
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=comparison, x="model", y="macro_f1", ax=axes[0])
sns.barplot(data=comparison, x="model", y="accuracy", ax=axes[1])
sns.barplot(data=comparison, x="model", y="latency_ms", ax=axes[2])
for ax in axes:
    ax.tick_params(axis="x", rotation=25)
    ax.grid(True, alpha=0.2)
fig.tight_layout()
fig.savefig(Path(CFG["output_dir"]) / "side_by_side_comparison.png", dpi=160)
plt.show()

## Explainability

The spectrogram models use Grad-CAM and LIME-style time-frequency perturbation. The waveform model uses segment occlusion because it consumes raw waveform embeddings.

In [ ]:
def first_correct_example(run):
    out = run["output"]
    for cls in range(NUM_CLASSES):
        idxs = np.where((out["labels"] == cls) & (out["preds"] == cls))[0]
        if len(idxs):
            sample = test_s[int(idxs[0])]
            spec, wave, label, path, source = AudioComparisonDataset([sample])[0]
            return spec, wave, label, path
    sample = test_s[0]
    spec, wave, label, path, source = AudioComparisonDataset([sample])[0]
    return spec, wave, label, path

def gradcam_spectrogram(name, model, spec, wave, label):
    if not getattr(model, "cam_layer", None):
        return None
    model.eval()
    acts, grads = [], []
    def fhook(module, inp, out):
        acts.append(out.detach())
    def bhook(module, gin, gout):
        grads.append(gout[0].detach())
    h1 = model.cam_layer.register_forward_hook(fhook)
    h2 = model.cam_layer.register_full_backward_hook(bhook)
    x = spec.unsqueeze(0).to(DEVICE)
    w = wave.unsqueeze(0).to(DEVICE)
    model.zero_grad()
    logits = model(x, w)
    score = logits[0, int(label)]
    score.backward()
    h1.remove()
    h2.remove()
    if not acts or not grads:
        return None
    weights = grads[0].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * acts[0]).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=(CFG["image_size"], CFG["image_size"]), mode="bilinear", align_corners=False)[0, 0]
    cam = cam.cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(spec[0].numpy(), origin="lower", cmap="magma")
    ax.imshow(cam, origin="lower", cmap="jet", alpha=0.38)
    ax.set_title(f"{name} Grad-CAM: {TARGET_CLASSES[int(label)]}")
    ax.axis("off")
    path = Path(CFG["xai_dir"]) / f"{name}_gradcam.png"
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    return str(path)

def spec_lime(name, model, spec, wave, label, grid=7):
    model.eval()
    cells = grid * grid
    xs, ys = [], []
    with torch.no_grad():
        for _ in range(CFG["lime_masks"]):
            mask_vec = np.random.binomial(1, 0.65, cells).astype(np.float32)
            mask = torch.tensor(mask_vec.reshape(grid, grid)).unsqueeze(0).unsqueeze(0)
            mask = F.interpolate(mask, size=(CFG["image_size"], CFG["image_size"]), mode="nearest").squeeze(0)
            masked = spec * mask
            prob = F.softmax(model(masked.unsqueeze(0).to(DEVICE), wave.unsqueeze(0).to(DEVICE)), dim=1)[0, int(label)].item()
            xs.append(mask_vec)
            ys.append(prob)
    ridge = Ridge(alpha=1.0).fit(np.array(xs), np.array(ys))
    heat = ridge.coef_.reshape(grid, grid)
    heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
    heat = F.interpolate(torch.tensor(heat).unsqueeze(0).unsqueeze(0).float(), size=(CFG["image_size"], CFG["image_size"]), mode="bilinear", align_corners=False)[0, 0].numpy()
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(spec[0].numpy(), origin="lower", cmap="magma")
    ax.imshow(heat, origin="lower", cmap="viridis", alpha=0.42)
    ax.set_title(f"{name} LIME-style attribution")
    ax.axis("off")
    path = Path(CFG["xai_dir"]) / f"{name}_lime_style.png"
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.show()
    return str(path)

def waveform_occlusion(name, model, spec, wave, label, chunks=16):
    model.eval()
    with torch.no_grad():
        base = F.softmax(model(spec.unsqueeze(0).to(DEVICE), wave.unsqueeze(0).to(DEVICE)), dim=1)[0, int(label)].item()
        drops = []
        length = wave.numel()
        step = length // chunks
        for i in range(chunks):
            masked = wave.clone()
            masked[i * step:(i + 1) * step] = 0
            prob = F.softmax(model(spec.unsqueeze(0).to(DEVICE), masked.unsqueeze(0).to(DEVICE)), dim=1)[0, int(label)].item()
            drops.append(base - prob)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(np.arange(chunks), drops)
    ax.set_title(f"{name} waveform occlusion")
    ax.set_xlabel("time segment")
    ax.set_ylabel("probability drop")
    fig.tight_layout()
    path = Path(CFG["xai_dir"]) / f"{name}_waveform_occlusion.png"
    fig.savefig(path, dpi=160)
    plt.show()
    return str(path)

XAI_FILES = {}
for name, run in valid_runs.items():
    spec, wave, label, source_path = first_correct_example(run)
    files = []
    if name == "candidate_panns_cnn14":
        files.append(waveform_occlusion(name, run["model"], spec, wave, label))
    else:
        files.append(gradcam_spectrogram(name, run["model"], spec, wave, label))
        files.append(spec_lime(name, run["model"], spec, wave, label))
    XAI_FILES[name] = [x for x in files if x]
with open(Path(CFG["xai_dir"]) / "xai_manifest.json", "w", encoding="utf-8") as f:
    json.dump(XAI_FILES, f, indent=2)
xai_rows = []
for model_name, files in XAI_FILES.items():
    for file_path in files:
        xai_rows.append({"model": model_name, "artifact": Path(file_path).name, "path": file_path})
xai_df = pd.DataFrame(xai_rows)
display(Markdown("### XAI artifact gallery"))
display(xai_df)
for model_name, files in XAI_FILES.items():
    display(Markdown(f"#### {model_name}"))
    for file_path in files:
        display(Image(filename=file_path))

## Final Selection and Output Files

In [ ]:
summary = {
    "best_model_for_next_notebook": BEST_MODEL_NAME,
    "final_vf_candidate": "final_vf_resnet34",
    "comparison": comparison.to_dict("records"),
    "dataset_audit": DATA_AUDIT,
    "xai_displayed": True,
    "output_dir": CFG["output_dir"]
}
with open(Path(CFG["output_dir"]) / "final_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
display(Markdown("### Final selection summary"))
display(pd.DataFrame(summary["comparison"]))
print(f"Final VF candidate: final_vf_resnet34")
print(f"Best model for later tuning: {BEST_MODEL_NAME}")

In [ ]:
display(Markdown("### Notebook output is displayed above"))
display(Markdown("The comparison table, learning curves, classification reports, confusion matrices, calibration plots, and XAI views are all rendered inside this notebook. Saved artifacts are optional Kaggle outputs, not required to inspect the comparison."))